In [1]:
import torch

if torch.cuda.is_available():
    print("CUDA device is available.")
else:
    print("CUDA device is not available.")

CUDA device is not available.


In [2]:
import nltk
import pandas as pd
import numpy as np
import fitz
from tqdm import tqdm
import re
from sentence_transformers import SentenceTransformer
import os
import pickle
import nltk
from keras.preprocessing.sequence import pad_sequences

# Deep Learning Model with Keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score

c:\Users\hp\Desktop\coding\SUPERVISED\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt_tab')
lemmatizer=WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [4]:
def preprocess_text(text):
    text = text.lower()
    text = re.sub('[^a-zA-Z]', ' ', text)
    tokens = word_tokenize(text)
    tokens = [lemmatizer.lemmatize(t) for t in tokens if t not in stop_words]
    return ' '.join(tokens)

In [6]:
def extract_text_from_pdf(pdf_path):
    doc = fitz.open(pdf_path)
    text = ""
    for page in doc:
        text += page.get_text() 
    doc.close()
    return text

Why LSTM cannot use SentenceTransformer embeddings
SentenceTransformer gives:
- One vector per document (shape: 768)
- No sequential structure
- No word order
- No relationship between tokens
LSTM requires:
A sequence of vectors (shape: sequence_length × embedding_dim)
So the shapes are incompatible.

In [7]:
def process_all_pdfs(base_directory):
    print(f"Processing PDFs from '{base_directory}'...")
    documents, labels = [], []
    for class_name in os.listdir(base_directory):
        class_path = os.path.join(base_directory, class_name)
        if os.path.isdir(class_path):
            print(f"  Loading class: {class_name}")
            for filename in tqdm(os.listdir(class_path), desc=f"  Processing {class_name}"):
                if filename.lower().endswith(".pdf"):
                    path = os.path.join(class_path, filename)
                    raw_text = extract_text_from_pdf(path)
                    preprocessed_text = preprocess_text(raw_text)
                    if preprocessed_text:
                        documents.append(preprocessed_text)
                        labels.append(class_name)
    return documents, labels

In [8]:
TRAINING_DOCS_DIR = r"C:\Users\hp\Desktop\coding\SUPERVISED\DATASET"
documents, labels = process_all_pdfs(TRAINING_DOCS_DIR)
print(f"Loaded {len(documents)} documents")
print(f"Labels: {set(labels)}")

Processing PDFs from 'C:\Users\hp\Desktop\coding\SUPERVISED\DATASET'...
  Loading class: automobiles


  Processing automobiles: 100%|██████████| 13/13 [00:09<00:00,  1.32it/s]


  Loading class: business


  Processing business: 100%|██████████| 510/510 [00:00<00:00, 517689.99it/s]


  Loading class: cybersecurity


...

  Loading class: sport


  Processing sport: 100%|██████████| 511/511 [00:00<00:00, 503119.56it/s]


  Loading class: tech


  Processing tech: 100%|██████████| 401/401 [00:00<00:00, 403337.15it/s]

Loaded 74 documents
Labels: {'financial', 'automobiles', 'cybersecurity', 'scientific_publication', 'market', 'HR'}


In [9]:
min(len(documents[i].split()) for i in range(len(documents)))

251

In [10]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    documents, labels, test_size=0.2, random_state=42
)

In [11]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

max_words = 50000 # vocabulary size: The tokenizer will keep only the top 50,000 most frequent words.
max_len = 1500 # maximum length: the tokenizer will read up to 1500 words from each document.

tokenizer = Tokenizer(num_words=max_words)
tokenizer.fit_on_texts(X_train)
X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)
X_train_pad = pad_sequences(X_train_seq, maxlen=max_len)
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len)

In [12]:
X_train_pad.shape, X_test_pad.shape

((59, 1500), (15, 1500))

In [13]:
from sklearn.preprocessing import LabelEncoder
from keras.utils import to_categorical

label_encoder = LabelEncoder()
y_train_int = label_encoder.fit_transform(y_train)
y_test_int = label_encoder.transform(y_test)
num_classes = len(label_encoder.classes_)
y_train_cat = to_categorical(y_train_int, num_classes)
y_test_cat = to_categorical(y_test_int, num_classes)


In [14]:
y_test_int

array([1, 5, 2, 1, 3, 5, 1, 3, 1, 4, 5, 3, 1, 0, 1])

In [15]:
y_test_cat

array([[0., 1., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 1.],
       [0., 0., 1., 0., 0., 0.],
       [0., 1., 0., 0., 0., 0.],
       [0., 0., 0., 1., 0., 0.],
       [0., 0., 0., 0., 0., 1.],
       [0., 1., 0., 0., 0., 0.],
       [0., 0., 0., 1., 0., 0.],
       [0., 1., 0., 0., 0., 0.],
       [0., 0., 0., 0., 1., 0.],
       [0., 0., 0., 0., 0., 1.],
       [0., 0., 0., 1., 0., 0.],
       [0., 1., 0., 0., 0., 0.],
       [1., 0., 0., 0., 0., 0.],
       [0., 1., 0., 0., 0., 0.]])

In [16]:
y_train_cat.shape, y_test_cat.shape

((59, 6), (15, 6))

In [17]:
from keras.models import Sequential
from keras.layers import Embedding, LSTM, Bidirectional, Dropout, Dense

model = Sequential()
model.add(Embedding(input_dim=max_words, output_dim=128, input_length=max_len))
model.add(LSTM(128, return_sequences=True))
model.add(LSTM(64))
model.add(Dense(32, activation='relu'))
model.add(Dense(num_classes, activation='softmax'))

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

c:\Users\hp\Desktop\coding\SUPERVISED\myenv\Lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [18]:
history = model.fit(
    X_train_pad,
    y_train_cat,
    epochs=10,
    batch_size=32,
    validation_split=0.2
)
model.summary()


Epoch 1/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 9s 2s/step - accuracy: 0.1277 - loss: 1.7923 - val_accuracy: 0.0833 - val_loss: 1.7938
Epoch 2/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 3s 1s/step - accuracy: 0.2766 - loss: 1.7794 - val_accuracy: 0.0833 - val_loss: 1.7967
Epoch 3/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 3s 1s/step - accuracy: 0.2340 - loss: 1.7613 - val_accuracy: 0.0833 - val_loss: 1.7991
Epoch 4/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 3s 1s/step - accuracy: 0.2340 - loss: 1.7368 - val_accuracy: 0.0833 - val_loss: 1.8063
Epoch 5/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 3s 1s/step - accuracy: 0.2340 - loss: 1.6897 - val_accuracy: 0.0833 - val_loss: 1.8223
Epoch 6/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 3s 1s/step - accuracy: 0.2553 - loss: 1.6097 - val_accuracy: 0.0833 - val_loss: 1.8344
Epoch 7/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 3s 1s/step - accuracy: 0.4681 - loss: 1.4419 - val_accuracy: 0.0833 - val_loss: 1.9360
Epoch 8/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 3s 1s/step - accuracy: 0.6170 - loss: 1.3563 - val_accuracy: 0.2500 - val_loss: 1.6987
Epoch 9/10
2/2 ━

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 1500, 128)      │     6,400,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 1500, 128)      │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 6)              │           198 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 19,749,812 (75.34 MB)

 Trainable params: 6,583,270 (25.11 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 13,166,542 (50.23 MB)

In [19]:
loss, acc = model.evaluate(X_test_pad, y_test_cat)
print("Test Accuracy:", acc)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 404ms/step - accuracy: 0.2667 - loss: 1.7449
Test Accuracy: 0.2666666805744171


In [20]:
import pickle

model.save('lstm_model.keras')

with open('tokenizer.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)

with open('label_encoder.pkl', 'wb') as f:
    pickle.dump(label_encoder, f)

print("LSTM model saved successfully!")

LSTM model saved successfully!


## Chunking with bilstms

In [21]:
def chunk_text(text, chunk_size=300):
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size):
        chunk = " ".join(words[i:i + chunk_size])
        chunks.append(chunk)
    return chunks

In [22]:
doc = "insurance automobile claim damage repair" 
chunks = chunk_text(doc, chunk_size=3)
chunks

['insurance automobile claim', 'damage repair']

In [23]:
def prepare_chunks(documents, tokenizer, max_len):
    all_docs_chunks = []
    for doc in documents:
        chunks = chunk_text(doc, chunk_size=300)
        seq = tokenizer.texts_to_sequences(chunks)
        padded = pad_sequences(seq, maxlen=max_len)
        all_docs_chunks.append(padded)
    return all_docs_chunks


In [24]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional
max_words = 50000 # 50,000 most important words 
max_len = 300  # chunk length
chunk_model = Sequential()
chunk_model.add(Embedding(max_words, 128, input_length=max_len))
chunk_model.add(Bidirectional(LSTM(128)))
chunk_model.add(Dropout(0.5))
chunk_model.add(Dense(128, activation='relu'))
chunk_model.add(Dropout(0.5))
chunk_model.add(Dense(num_classes, activation='softmax'))

chunk_model.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

In [25]:
X_chunks = []
y_chunks = []
X_train_chunks = prepare_chunks(X_train, tokenizer, max_len)
for i, chunks in enumerate(X_train_chunks):
    for chunk in chunks:
        X_chunks.append(chunk)
        y_chunks.append(y_train_cat[i])
X_chunks = np.array(X_chunks)
y_chunks = np.array(y_chunks)
X_chunks.shape, y_chunks.shape

((2348, 300), (2348, 6))

In [26]:
chunk_model.fit(
    X_chunks,
    y_chunks,
    batch_size=32,
    epochs=10,
    validation_split=0.2
)


Epoch 1/10
59/59 ━━━━━━━━━━━━━━━━━━━━ 19s 232ms/step - accuracy: 0.5000 - loss: 1.2821 - val_accuracy: 0.5021 - val_loss: 1.9918
Epoch 2/10
59/59 ━━━━━━━━━━━━━━━━━━━━ 13s 212ms/step - accuracy: 0.7614 - loss: 0.6524 - val_accuracy: 0.5915 - val_loss: 0.9873
Epoch 3/10
59/59 ━━━━━━━━━━━━━━━━━━━━ 13s 221ms/step - accuracy: 0.8775 - loss: 0.3362 - val_accuracy: 0.8255 - val_loss: 0.7132
Epoch 4/10
59/59 ━━━━━━━━━━━━━━━━━━━━ 13s 226ms/step - accuracy: 0.9425 - loss: 0.1896 - val_accuracy: 0.8043 - val_loss: 0.7502
Epoch 5/10
59/59 ━━━━━━━━━━━━━━━━━━━━ 13s 224ms/step - accuracy: 0.9505 - loss: 0.1510 - val_accuracy: 0.8149 - val_loss: 0.7021
Epoch 6/10
59/59 ━━━━━━━━━━━━━━━━━━━━ 13s 224ms/step - accuracy: 0.9830 - loss: 0.0777 - val_accuracy: 0.8511 - val_loss: 0.5621
Epoch 7/10
59/59 ━━━━━━━━━━━━━━━━━━━━ 14s 229ms/step - accuracy: 0.9952 - loss: 0.0232 - val_accuracy: 0.8957 - val_loss: 0.4669
Epoch 8/10
59/59 ━━━━━━━━━━━━━━━━━━━━ 13s 228ms/step - accuracy: 0.9963 - loss: 0.0147 - val_accu

In [27]:
loss, acc = chunk_model.evaluate(X_test_pad, y_test_cat)
print("Test Accuracy:", acc)


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 937ms/step - accuracy: 1.0000 - loss: 0.0519
Test Accuracy: 1.0


In [28]:

model.save('chunk_model.keras')